# Stockholm Bus Deadhead Analysis

This notebook fetches GTFS-RT vehicle position data from the Trafiklab KoDa API and analyzes deadhead (tomkörning) patterns for Stockholm bus routes.

**Works in:** Google Colab, GitHub Codespaces, or any local Jupyter environment.

## 1. Environment Setup

Installs dependencies and sets up the project path. Handles both Colab and Codespaces automatically.

In [ ]:
import os
import sys

# Detect environment
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Clone the repo into Colab
    REPO_URL = "https://github.com/HEVI-SE/Trafiklab.git"
    REPO_DIR = "/content/Trafiklab"
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)
    print(f"Colab: working directory set to {REPO_DIR}")
else:
    # Codespaces / local: ensure we're in the repo root
    notebook_dir = os.path.dirname(os.path.abspath("__file__"))
    if os.path.exists(os.path.join(notebook_dir, "config.py")):
        os.chdir(notebook_dir)
        sys.path.insert(0, notebook_dir)
    print(f"Local: working directory is {os.getcwd()}")

# Install dependencies
!pip install -q pandas requests py7zr plotly gtfs-realtime-bindings

print("Setup complete!")

## 2. Configuration

Set the date and hours you want to analyze. The API key has a default but you can override it.

In [ ]:
# ---- EDIT THESE ----
DATE = "2025-03-17"           # Date to analyze (YYYY-MM-DD)
HOURS = list(range(6, 22))    # Hours to fetch (6 AM to 9 PM)

# Optional: override the default API key
# os.environ["KODA_API_KEY"] = "your_key_here"

print(f"Will analyze: {DATE}, hours {HOURS[0]:02d}-{HOURS[-1]:02d}")

## 3. Load Static GTFS Schedule

Downloads the static GTFS data (routes, trips, stops, stop_times) for the selected date.

In [ ]:
import pandas as pd
from config import OPERATOR_MAPPING
from fetcher import load_static_gtfs, build_trip_lookup

routes, trips, stops, stop_times = load_static_gtfs(DATE)

print(f"Routes: {len(routes)}, Trips: {len(trips)}, Stops: {len(stops)}, Stop times: {len(stop_times)}")

## 4. Build Trip Lookup Table

Creates a lookup mapping trip_id to route, operator, headsign, first/last stop.

In [ ]:
operator_df = pd.DataFrame(OPERATOR_MAPPING)
trip_lookup = build_trip_lookup(trips, routes, operator_df, stop_times, stops)

print(f"Trip lookup: {len(trip_lookup)} trips")
print(f"Operators: {trip_lookup['operator'].value_counts().to_dict()}")
trip_lookup.head()

## 5. Fetch Vehicle Positions

Downloads GTFS-RT vehicle positions hour by hour and builds movement segments.

**Note:** This step can take 10-30 minutes depending on the number of hours. The KoDa API may return 202 (generating archive) and the code will retry automatically.

In [ ]:
from fetcher import fetch_vehicle_positions

segments = fetch_vehicle_positions(DATE, HOURS, trip_lookup)

print(f"\nTotal segments: {len(segments)}")
print(f"Unique vehicles: {segments['vehicle_id'].nunique()}")
print(f"Routes observed: {segments['route_short_name'].nunique()}")
segments.head(10)

## 6. Detect Deadheads (Tomkörningar)

Identifies periods where buses travel empty between trips.

In [ ]:
from analysis import build_observed_deadheads, build_planned_deadheads

# Observed deadheads (from real-time vehicle tracking)
observed = build_observed_deadheads(segments, stops)
print(f"Observed deadheads: {len(observed)}")

# Planned deadheads (from static GTFS schedule)
planned = build_planned_deadheads(trips, stop_times, stops, routes, operator_df)
print(f"Planned deadheads: {len(planned)}")

## 7. Save Results to CSV

Saves segments and deadheads to CSV files in the `data/` directory. Automatically deduplicates with any existing data.

In [ ]:
from csv_handler import save_segments, save_deadheads

save_segments(segments)

if not observed.empty:
    save_deadheads(observed)

if not planned.empty:
    save_deadheads(planned)

print("\nResults saved to data/ directory.")

## 8. Analysis Summary

Overview statistics of the collected data.

In [ ]:
from utils import classify_period

print("=" * 60)
print(f"ANALYSIS SUMMARY FOR {DATE}")
print("=" * 60)

# Segment stats
print(f"\n--- Vehicle Segments ---")
print(f"Total segments: {len(segments):,}")
print(f"Unique vehicles: {segments['vehicle_id'].nunique():,}")
print(f"Time range: {segments['start_time'].min()} to {segments['end_time'].max()}")

route_counts = segments[segments['route_short_name'] != 'Okänd tur']['route_short_name'].value_counts()
print(f"\nTop 10 routes by segment count:")
print(route_counts.head(10).to_string())

# Deadhead stats
if not observed.empty:
    print(f"\n--- Observed Deadheads ---")
    print(f"Total: {len(observed):,}")
    print(f"Avg duration: {observed['duration_min'].mean():.1f} min")
    print(f"Avg distance: {observed['move_m'].mean():.0f} m")
    print(f"\nBy traffic period:")
    print(observed['period'].value_counts().to_string())
    print(f"\nBy operator:")
    print(observed['operator'].value_counts().to_string())
else:
    print("\nNo observed deadheads detected.")

if not planned.empty:
    print(f"\n--- Planned Deadheads ---")
    print(f"Total: {len(planned):,}")
    print(f"\nBy operator:")
    print(planned['operator'].value_counts().to_string())
else:
    print("\nNo planned deadheads found.")

## 9. Visualizations

In [ ]:
import plotly.express as px

# Deadheads by operator
if not observed.empty:
    fig = px.histogram(
        observed, x="operator", color="period",
        title=f"Observed Deadheads by Operator & Traffic Period ({DATE})",
        labels={"operator": "Operator", "count": "Count", "period": "Traffic Period"},
        barmode="stack",
    )
    fig.update_layout(xaxis_categoryorder="total descending")
    fig.show()

In [ ]:
# Deadhead duration distribution
if not observed.empty:
    fig = px.histogram(
        observed, x="duration_min", nbins=30,
        title=f"Deadhead Duration Distribution ({DATE})",
        labels={"duration_min": "Duration (minutes)", "count": "Count"},
    )
    fig.show()

In [ ]:
# Top deadhead routes
if not observed.empty:
    top_routes = (
        observed.groupby(["prev_route", "next_route"])
        .agg(count=("vehicle_id", "size"), avg_duration=("duration_min", "mean"), avg_distance=("move_m", "mean"))
        .reset_index()
        .sort_values("count", ascending=False)
        .head(15)
    )
    top_routes["route_pair"] = top_routes["prev_route"] + " → " + top_routes["next_route"]
    top_routes["avg_duration"] = top_routes["avg_duration"].round(1)
    top_routes["avg_distance"] = (top_routes["avg_distance"] / 1000).round(1)  # km

    fig = px.bar(
        top_routes, x="route_pair", y="count",
        hover_data=["avg_duration", "avg_distance"],
        title=f"Top 15 Deadhead Route Pairs ({DATE})",
        labels={"route_pair": "Route Pair", "count": "Count", "avg_duration": "Avg Duration (min)", "avg_distance": "Avg Distance (km)"},
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

In [ ]:
# Deadhead timeline
if not observed.empty:
    timeline = observed.copy()
    timeline["hour"] = pd.to_datetime(timeline["deadhead_start"]).dt.hour
    hourly = timeline.groupby("hour").agg(
        count=("vehicle_id", "size"),
        total_km=("move_m", lambda x: (x.sum() / 1000).round(1)),
    ).reset_index()

    fig = px.bar(
        hourly, x="hour", y="count",
        hover_data=["total_km"],
        title=f"Deadheads by Hour of Day ({DATE})",
        labels={"hour": "Hour", "count": "Number of Deadheads", "total_km": "Total Distance (km)"},
    )
    fig.update_layout(xaxis_dtick=1)
    fig.show()

## 10. Generate HTML Report

Downloads a dark-themed HTML report with interactive dropdown tables for all deadheads, grouped by route pair and traffic period.

In [ ]:
from report import generate_html_report

report_path = generate_html_report(observed, planned, segments, DATE)

# Auto-download in Colab, or print path for local use
if IN_COLAB:
    from google.colab import files
    files.download(report_path)
    print("Download started!")
else:
    import webbrowser
    abs_path = os.path.abspath(report_path)
    print(f"Report saved to: {abs_path}")
    try:
        webbrowser.open(f"file://{abs_path}")
        print("Opened in browser.")
    except Exception:
        print("Open the file above in your browser to view the report.")

## 11. Explore Raw Data

Browse the raw dataframes interactively.

In [ ]:
# View observed deadheads
if not observed.empty:
    display_cols = ["operator", "prev_route", "next_route", "from_stop_observed", "to_stop_observed",
                    "deadhead_start", "duration_min", "move_m", "speed_kmh", "period"]
    available = [c for c in display_cols if c in observed.columns]
    observed[available].head(20)
else:
    print("No observed deadheads to show.")